# Fakeout Reversal (Trap Bars) on SPY
## Strategy Brief -- 3-5 plain-English sentences on signal, prediction, trade logic, results.
The Fakeout Reversal strategy aims to identify false breakouts in SPY, where price action initially suggests a trend continuation but then reverses direction. The strategy predicts that after a fake breakout, the price will reverse and move in the opposite direction. The trade logic involves entering a position when a fakeout is detected and exiting when a predefined target or stop-loss is hit. Results are evaluated based on historical performance metrics like CAGR, Sharpe ratio, and drawdowns.
## References
- https://www.fakeout.io/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the trading parameters and constants that will be used throughout the notebook. These include the lookback period for detecting fakeouts, thresholds for identifying reversals, and risk management rules.

In [ ]:
LOOKBACK_PERIOD = 20
FAKEOUT_THRESHOLD = 0.02
STOP_LOSS = 0.03
TAKE_PROFIT = 0.05
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

### PHASE 2 - Data Exploration
We will download historical SPY data using yfinance, calculate the necessary indicators to identify fakeouts, and visualize these indicators overlaid on the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate rolling high and low
data['Rolling_High'] = data['High'].rolling(window=LOOKBACK_PERIOD).max()
data['Rolling_Low'] = data['Low'].rolling(window=LOOKBACK_PERIOD).min()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['Rolling_High'], label='Rolling High', linestyle='--')
plt.plot(data['Rolling_Low'], label='Rolling Low', linestyle='--')
plt.title('SPY Price with Rolling High/Low')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
In this phase, we create the signal series based on fakeout detection logic, define entry and exit logic, and generate a position series that indicates when to be long or short.

In [ ]:
# Detect fakeouts
data['Fakeout_Signal'] = ((data['High'] > data['Rolling_High'] * (1 + FAKEOUT_THRESHOLD)) & (data['Close'] < data['Rolling_High'])) | \
                       ((data['Low'] < data['Rolling_Low'] * (1 - FAKEOUT_THRESHOLD)) & (data['Close'] > data['Rolling_Low']))

# Entry/Exit logic
data['Position'] = 0
data.loc[data['Fakeout_Signal'], 'Position'] = np.where(data['Close'] < data['Rolling_High'], 1, -1)

### PHASE 4 - Coding & Backtesting
We will shift the positions to avoid lookahead bias, calculate daily returns, and plot the equity curve to visualize the strategy's performance.

In [ ]:
# Shift positions
data['Position'] = data['Position'].shift(1)

# Calculate daily returns
data['Market_Returns'] = data['Close'].pct_change()
data['Strategy_Returns'] = data['Position'] * data['Market_Returns']

data['Equity_Curve'] = (1 + data['Strategy_Returns']).cumprod()

data[['Equity_Curve']].plot(figsize=(14, 7), title='Equity Curve')
plt.show()

### PHASE 5 - Performance Evaluation
We will compute key performance metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. These will be compared against a buy-and-hold strategy.

In [ ]:
def calculate_performance(data):
    # CAGR
    years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (data['Equity_Curve'].iloc[-1]) ** (1/years) - 1

    # Sharpe Ratio
    sharpe_ratio = data['Strategy_Returns'].mean() / data['Strategy_Returns'].std() * np.sqrt(252)

    # Sortino Ratio
    downside_std = data['Strategy_Returns'][data['Strategy_Returns'] < 0].std()
    sortino_ratio = data['Strategy_Returns'].mean() / downside_std * np.sqrt(252)

    # Max Drawdown
    roll_max = data['Equity_Curve'].cummax()
    daily_drawdown = data['Equity_Curve'] / roll_max - 1.0
    max_drawdown = daily_drawdown.min()

    # Calmar Ratio
    calmar_ratio = cagr / abs(max_drawdown)

    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

# Calculate performance
cagr, sharpe, sortino, calmar, max_dd = calculate_performance(data)

# Buy and Hold
buy_hold_cagr = (data['Close'].iloc[-1] / data['Close'].iloc[0]) ** (1/years) - 1

# Print comparison
eval_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [cagr, sharpe, sortino, calmar, max_dd],
    'Buy & Hold': [buy_hold_cagr, 'N/A', 'N/A', 'N/A', 'N/A']
})
print(eval_df)

### PHASE 6 - Deploy & Monitor
We will create a function that downloads the last 60 days of SPY data, computes today's signal, and prints the recommended position.

In [ ]:
def get_latest_signal():
    # Download last 60 days
data = yf.download('SPY', period='60d')

    # Calculate rolling high and low
data['Rolling_High'] = data['High'].rolling(window=LOOKBACK_PERIOD).max()
data['Rolling_Low'] = data['Low'].rolling(window=LOOKBACK_PERIOD).min()

    # Detect fakeouts
data['Fakeout_Signal'] = ((data['High'] > data['Rolling_High'] * (1 + FAKEOUT_THRESHOLD)) & (data['Close'] < data['Rolling_High'])) | \
                           ((data['Low'] < data['Rolling_Low'] * (1 - FAKEOUT_THRESHOLD)) & (data['Close'] > data['Rolling_Low']))

    # Determine today's position
    latest_signal = data['Fakeout_Signal'].iloc[-1]
    if latest_signal:
        position = 'Long' if data['Close'].iloc[-1] < data['Rolling_High'].iloc[-1] else 'Short'
    else:
        position = 'Neutral'

    print(f"Today's position: {position}")

# Get today's signal
get_latest_signal()